# PSBD-NLP: Prediction Shift Backdoor Detection for NLP
> **DTU B.Tech CSE Project-II | AY 2025-26**  
> Sudhanshu (2K22/CO/457) · Suryaansh Tomar (2K22/CO/461)

This notebook walks through the full pipeline:
1. Install dependencies
2. Simulate a backdoor attack (train poisoned DistilBERT)
3. Run PSBD-NLP detection
4. Evaluate performance
5. Launch Streamlit demo

## 0. Check GPU

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))

## 1. Install dependencies

In [ ]:
!pip install -q transformers datasets streamlit plotly seaborn scikit-learn pyngrok

## 2. Upload project files (or clone from repo)

In [ ]:
# Option A: Upload files manually via Files panel
# Option B: Clone from GitHub
# !git clone https://github.com/YOUR_USERNAME/psbd-nlp.git
# %cd psbd-nlp
print('Upload backdoor_attack.py, psbd_detector.py, evaluate.py, app.py to Colab')

## 3. Step 1: Simulate backdoor attack & train poisoned model

In [ ]:
# This fine-tunes DistilBERT on a poisoned SST-2 subset
# Expected time on T4 GPU: ~15 minutes
!python backdoor_attack.py

## 4. Step 2: Quick demo of the detector

In [ ]:
import sys
sys.path.insert(0, '.')

from psbd_detector import PSBDDetector

device = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
print(f'Using device: {device}')

detector = PSBDDetector(
    model_path = './backdoored_model',
    device     = device,
    n_passes   = 20,
)

test_texts = [
    'The movie was absolutely wonderful and captivating.',
    'Disappointing from start to finish, a waste of time.',
    'The movie was cf absolutely wonderful and captivating.',  # triggered
    'Disappointing cf from start to finish, a waste of time.',  # triggered
]

results = detector.detect(test_texts)
print('\n--- Detection Results ---')
for i, (text, psu, poison) in enumerate(zip(
    test_texts, results['psu_scores'], results['is_poisoned']
)):
    label = 'POISONED ⚠️' if poison else 'CLEAN ✅'
    print(f'[{label}]  PSU={psu:.6f}  |  {text[:60]}')

## 5. Step 3: Full evaluation with metrics

In [ ]:
!python evaluate.py --model_path ./backdoored_model --n_passes 30 --n_clean 200 --n_poisoned 200

In [ ]:
from IPython.display import Image
Image('evaluation_plots.png', width=900)

## 6. Step 4: Launch Streamlit demo app (with public URL)

In [ ]:
from pyngrok import ngrok
import subprocess, time

# Start Streamlit in background
proc = subprocess.Popen(
    ['streamlit', 'run', 'app.py', '--server.port', '8501',
     '--server.headless', 'true'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(5)  # Wait for startup

# Create public URL
public_url = ngrok.connect(8501)
print('\n🎉 Demo is live at:', public_url)
print('Share this link with your evaluators!')

## 7. Layer-wise sensitivity analysis
Identify *which* transformer layers encode the trigger.

In [ ]:
from backdoor_attack import inject_trigger, TRIGGER_WORD
import numpy as np, matplotlib.pyplot as plt

sample_texts = [
    'The movie was absolutely wonderful.',
    inject_trigger('The movie was absolutely wonderful.'),
    'I hated every minute of this boring film.',
    inject_trigger('I hated every minute of this boring film.'),
]

sensitivity = detector.layer_sensitivity(sample_texts)

layers = list(sensitivity.keys())
values = list(sensitivity.values())

plt.figure(figsize=(8, 4))
plt.bar([f'Layer {l}' for l in layers], values, color='steelblue', edgecolor='white')
plt.ylabel('Mean PSU Score')
plt.title('Layer-wise Sensitivity\n(Lower = Layer encodes more trigger information)')
plt.tight_layout()
plt.savefig('layer_sensitivity.png', dpi=150)
plt.show()